In [ ]:
from openpiv import tools, pyprocess, validation, filters, scaling

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import cv2

from pathlib import Path

In [ ]:
def get_frame_gray(cap, idx):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    success, frame = cap.read()

    if success:
        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        return gray_frame
    else:
        raise Exception('Could not get frame')
    

In [ ]:
vid_fn = '/Users/michaellavender/Documents/BUSPC/optical flow/OpenPIV/video-proc-guass5.mp4'
cap = cv2.VideoCapture(vid_fn)
fps = cap.get(cv2.CAP_PROP_FPS)
t1 = 10
# t2 = 10.5

i1 = int(t1 * fps)
# i2 = int(t2 * fps)

f1 = get_frame_gray(cap, i1)
f2 = get_frame_gray(cap, i1 + 5)

cap.release()
cv2.destroyAllWindows()

plt.imshow(f1, 'gray')
plt.imshow(f2, 'gray')

In [ ]:
winsize = 32 # pixels, interrogation window size in frame A
searchsize = 38  # pixels, search area size in frame B
overlap = 19 # pixels, 50% overlap
# dt = t2 - t1 # sec, time interval between the two frames
dt = 5 / fps

u0, v0, sig2noise = pyprocess.extended_search_area_piv(
    f1.astype(np.int32),
    f2.astype(np.int32),
    window_size=winsize,
    overlap=overlap,
    dt=dt,
    search_area_size=searchsize,
    sig2noise_method='peak2peak',
)

In [ ]:
x, y = pyprocess.get_coordinates(
    image_size=f1.shape,
    search_area_size=searchsize,
    overlap=overlap,
)

In [ ]:
plt.hist(sig2noise.flatten(), bins=30);

In [ ]:
invalid_mask = validation.sig2noise_val(
    sig2noise,
    threshold = 1.05,
)

In [ ]:
u2, v2 = filters.replace_outliers(
    u0, v0,
    invalid_mask,
    method='localmean',
    max_iter=3,
    kernel_size=3,
)

In [ ]:
# FROM TUTORIAL
# convert x,y to mm
# convert u,v to mm/sec
SF = 1
x, y, u3, v3 = scaling.uniform(
    x, y, u2, v2,
    scaling_factor = SF,  # 96.52 pixels/millimeter
)

# 0,0 shall be bottom left, positive rotation rate is counterclockwise
x, y, u3, v3 = tools.transform_coordinates(x, y, u3, v3)

u3 = np.ma.filled(np.ma.asarray(u3).astype(float), np.nan)   # <-- add
v3 = np.ma.filled(np.ma.asarray(v3).astype(float), np.nan)   # <-- add


In [ ]:
tools.save('aurora.txt' , x, y, u3, v3, invalid_mask)
cv2.imwrite('f1.png', f1);
cv2.imwrite('f2.png', f2);

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
tools.display_vector_field(
    Path('aurora.txt'),
    ax=ax, scaling_factor=SF,
    scale=np.nanmedian(np.hypot(u3, v3)) * 40, # scale defines here the arrow length
    width=0.0035, # width is the thickness of the arrow
    on_img=True, # overlay on the image
    image_name= 'f1.png',
);

# From Claude

In [ ]:
"""
PIV analysis of a video file with OpenPIV.

OpenPIV works on pairs of 2D arrays; this script decodes a video and feeds
consecutive frame pairs through the standard OpenPIV pipeline.

    pip install openpiv opencv-python numpy matplotlib

Usage:
    python piv_video.py flow.mp4 --winsize 32 --overlap 16 --searchsize 38
"""

import argparse
from pathlib import Path

import cv2
import numpy as np
from openpiv import pyprocess, validation, filters, scaling, tools


def frame_pairs(path, step=1, mode="rolling", start=0, stop=None):
    """Yield (index, frame_a, frame_b, dt) from a video.

    mode="rolling"  -> pairs (0,step), (1,1+step), (2,2+step) ...  time-resolved
    mode="disjoint" -> pairs (0,step), (2*step,3*step) ...         independent samples
    """
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise IOError(f"cannot open {path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or np.isnan(fps):
        raise ValueError("no fps in container metadata; pass dt explicitly")
    dt = step / fps

    # Decode once into memory if the clip is small; otherwise stream.
    # Streaming version below keeps only a short buffer.
    buf = []
    idx = -1
    pair_i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        idx += 1
        if idx < start:
            continue
        if stop is not None and idx > stop:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.int32)
        buf.append(gray)
        if len(buf) > step + 1:
            buf.pop(0)
        if len(buf) == step + 1:
            yield pair_i, buf[0], buf[-1], dt
            pair_i += 1
            if mode == "disjoint":
                buf.clear()
    cap.release()


def piv_pair(frame_a, frame_b, dt, winsize=32, overlap=16, searchsize=38,
             s2n_threshold=1.05, scaling_factor=1.0):
    """Run one OpenPIV pass on a frame pair. Returns x, y, u, v, invalid_mask."""
    u0, v0, s2n = pyprocess.extended_search_area_piv(
        frame_a, frame_b,
        window_size=winsize,
        overlap=overlap,
        dt=dt,
        search_area_size=searchsize,
        sig2noise_method="peak2peak",
    )

    x, y = pyprocess.get_coordinates(
        image_size=frame_a.shape,
        search_area_size=searchsize,
        overlap=overlap,
    )

    invalid_mask = validation.sig2noise_val(s2n, threshold=s2n_threshold)

    u, v = filters.replace_outliers(
        u0, v0, invalid_mask,
        method="localmean", max_iter=3, kernel_size=3,
    )

    # scaling_factor is pixels per metre; 1.0 leaves results in px/s
    x, y, u, v = scaling.uniform(x, y, u, v, scaling_factor=scaling_factor)
    x, y, u, v = tools.transform_coordinates(x, y, u, v)
    return x, y, u, v, invalid_mask


def main():
    p = argparse.ArgumentParser()
    p.add_argument("video")
    p.add_argument("--winsize", type=int, default=32)
    p.add_argument("--overlap", type=int, default=16)
    p.add_argument("--searchsize", type=int, default=38)
    p.add_argument("--step", type=int, default=1,
                   help="frames between a and b; raise it if displacement is tiny")
    p.add_argument("--mode", choices=["rolling", "disjoint"], default="rolling")
    p.add_argument("--start", type=int, default=0)
    p.add_argument("--stop", type=int, default=None)
    p.add_argument("--scaling-factor", type=float, default=1.0,
                   help="pixels per metre; 1.0 keeps px/s")
    p.add_argument("--s2n", type=float, default=1.05)
    p.add_argument("--outdir", default="piv_out")
    p.add_argument("--save-txt", action="store_true",
                   help="also write one OpenPIV .txt per pair")
    args = p.parse_args()

    outdir = Path(args.outdir)
    outdir.mkdir(exist_ok=True)

    us, vs, x, y = [], [], None, None

    for i, a, b, dt in frame_pairs(args.video, step=args.step, mode=args.mode,
                                   start=args.start, stop=args.stop):
        x, y, u, v, mask = piv_pair(
            a, b, dt,
            winsize=args.winsize,
            overlap=args.overlap,
            searchsize=args.searchsize,
            s2n_threshold=args.s2n,
            scaling_factor=args.scaling_factor,
        )
        us.append(u)
        vs.append(v)
        if args.save_txt:
            tools.save(str(outdir / f"pair_{i:05d}.txt"), x, y, u, v, mask)
        print(f"pair {i:5d}  dt={dt:.5f}s  "
              f"|V|mean={np.nanmean(np.hypot(u, v)):.3f}  "
              f"invalid={mask.mean():.1%}")

    if not us:
        print("no pairs produced -- check --start/--stop/--step")
        return

    U = np.stack(us)          # shape (n_pairs, ny, nx)
    V = np.stack(vs)

    # Time-average field and per-vector RMS fluctuation (Reynolds decomposition)
    U_mean, V_mean = np.nanmean(U, axis=0), np.nanmean(V, axis=0)
    u_rms = np.nanstd(U, axis=0)
    v_rms = np.nanstd(V, axis=0)

    np.savez_compressed(
        outdir / "piv_series.npz",
        x=x, y=y, U=U, V=V,
        U_mean=U_mean, V_mean=V_mean, u_rms=u_rms, v_rms=v_rms,
    )
    print(f"\nsaved {U.shape[0]} fields of {U.shape[1]}x{U.shape[2]} "
          f"-> {outdir / 'piv_series.npz'}")

    # Quick look at the mean field
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(8, 6))
        speed = np.hypot(U_mean, V_mean)
        ax.imshow(speed, origin="upper",
                  extent=[x.min(), x.max(), y.min(), y.max()], alpha=0.8)
        ax.quiver(x, y, U_mean, V_mean, color="w", scale_units="xy")
        ax.set_title("time-averaged velocity")
        fig.savefig(outdir / "mean_field.png", dpi=150, bbox_inches="tight")
        print(f"wrote {outdir / 'mean_field.png'}")
    except ImportError:
        pass


if __name__ == "__main__":
    main()